In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import root_mean_squared_error
import time
#import pygwalker as pyg
from datetime import datetime
import os
#from ydata_profiling import ProfileReport
import csv

In [178]:
# train_df = pd.read_csv('../artifacts/train.csv')
# test_df = pd.read_csv('../artifacts/test.csv')
df = pd.read_csv('../artifacts/raw.csv')

C:\Users\Dhvanish\AppData\Local\Temp\ipykernel_30796\13393110.py:3: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../artifacts/raw.csv')


## Imputing missing values ##

In [179]:
df['CompetitionDistance'].fillna(0,inplace=True)
#train_df.isnull().sum()

In [180]:
df['Date'] = pd.to_datetime(df['Date'])

df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day

df['CompetitionOpen_missing'] = df['CompetitionOpenSinceYear'].isna().astype(int)
df['CompetitionOpenSinceMonth'].fillna(1,inplace=True)
df['CompetitionOpenSinceYear'].fillna(df['Year'].min(),inplace=True)
df['CompetitionOpen'] = 12 * (df['Year'] - df['CompetitionOpenSinceYear']) + (df['Month'] - 
                                                                              df['CompetitionOpenSinceMonth'])
df['CompetitionOpen'] = df['CompetitionOpen'].apply(lambda x: max(x, 0))


In [181]:
df['WeekOfYear'] = df['Date'].dt.isocalendar().week
df['Promo2OpenSinceMonths'] = 12 * (df['Year'] - df['Promo2SinceYear']) + (df['WeekOfYear'] - 
                                                                           df['Promo2SinceWeek']) / 4.0
df['Promo2OpenSinceMonths'] = df['Promo2OpenSinceMonths'].apply(lambda x: max(x, 0) if pd.notnull(x) else 0)
df.loc[df['Promo2'] == 0, 'Promo2OpenSinceMonths'] = 0
month_map = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
             7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
df['MonthStr'] = df['Month'].map(month_map)
promo_months = df['PromoInterval'].fillna('').str.split(',')
df['IsPromoMonth'] = [
    1 if m in months else 0 
    for m, months in zip(df['MonthStr'], promo_months)
]


In [182]:
df.drop(['Store','CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear','Promo2SinceWeek','Promo2SinceYear','PromoInterval','MonthStr'], axis=1, inplace=True)

## Feature Engineering ##

In [183]:
df['StateHoliday'] = np.where((df['StateHoliday'] == '0') | (df['StateHoliday'] == 0),0,1)

In [192]:
walker = pyg.walk(df)

Box(children=(HTML(value='\n<div id="ifr-pyg-00065a9616067839r7JVneNDYEApdcfK" style="height: auto">\n    <hea…

In [185]:
df.sample(20)

,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,Promo2,Year,Month,Day,CompetitionOpen_missing,CompetitionOpen,WeekOfYear,Promo2OpenSinceMonths,IsPromoMonth
422605,3,2014-06-18,8791,675,1,1,0,1,d,a,260.0,1,2014,6,18,1,17.0,25,43.25,0
466832,5,2014-05-09,6067,653,1,1,0,0,a,c,30030.0,0,2014,5,9,0,42.0,19,0.00,0
490846,5,2014-04-18,0,0,0,1,1,0,d,c,270.0,1,2014,4,18,0,14.0,16,18.00,1
24920,4,2015-07-09,4789,610,1,0,0,1,a,a,460.0,1,2015,7,9,0,8.0,28,23.25,0
325651,6,2014-09-27,9227,1096,1,0,0,0,a,c,1070.0,0,2014,9,27,0,49.0,39,0.00,0
176935,1,2015-02-23,5913,479,1,0,0,0,d,c,9820.0,0,2015,2,23,1,25.0,9,0.00,0
552061,6,2014-02-22,8138,825,1,0,0,0,d,a,2110.0,0,2014,2,22,0,95.0,8,0.00,0
812559,3,2013-07-03,10498,1044,1,1,0,0,a,c,8260.0,0,2013,7,3,1,6.0,27,0.00,0
383225,7,2014-07-27,0,0,0,0,0,0,a,c,350.0,1,2014,7,27,0,79.0,30,31.25,1
102903,4,2015-04-30,8653,1004,1,1,0,0,a,a,13140.0,1,2015,4,30,1,27.0,18,49.00,1


In [186]:
num_col=['Customers','CompetitionDistance','CompetitionOpen','Promo2OpenSinceMonths','DayOfWeek','Month','Day']
cat_col = ['StoreType','Assortment','Year']

In [188]:
df = df.sort_values('Date')

cutoff_date = '2015-06-01'

train = df[df['Date'] < cutoff_date]
valid = df[df['Date'] >= cutoff_date]

X_train = train.drop(['Sales', 'Date'], axis=1)
y_train = train['Sales']

X_test = valid.drop(['Sales', 'Date'], axis=1)
y_test = valid['Sales']

In [189]:
preprocessor = ColumnTransformer([
    ('scl',StandardScaler(),num_col),
    ('ohe',OneHotEncoder(drop='first'),cat_col)    
])

In [190]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [ ]:
results_file = 'rmsep_score.csv'
file_exists = os.path.isfile(results_file)

models = {
    "XGBRegressor" : XGBRegressor(tree_method='hist',n_jobs=-1),
    'RandomForestRegressor': RandomForestRegressor(n_estimators=50,max_depth=15,n_jobs=-1),
    'LinearRegression': LinearRegression(),
    'LGBMRegressor': LGBMRegressor(n_jobs=-1)
}

results = []

for model_name,model in models.items():
    y_test_mean = np.mean(y_test)

    training_start = time.perf_counter()
    model.fit(X_train,y_train)
    training_stop = time.perf_counter()
    training_time_taken = training_stop - training_start

    prediction_start = time.perf_counter()
    prediction = model.predict(X_test)
    prediction_stop = time.perf_counter()
    prediction_time_taken = prediction_stop-prediction_start
    rmse = root_mean_squared_error(y_test,prediction)
    rmsep = rmse/y_test_mean

    print(f'{model_name}: {rmsep*100:.2f}%\n')
    print(f'Training time taken for {model_name}: {training_time_taken:.4f}\n')
    print(f'prediction time taken for {model_name}: {prediction_time_taken:.4f}\n')

    results.append({
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'model_name': model_name,
        'rmse': rmse,
        'rmsep_percent': rmsep * 100,
        'prediction_time_taken_sec': prediction_time_taken,
        'training_time_taken_sec': training_time_taken
    })

with open(results_file, 'a', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['timestamp', 'model_name', 'rmse', 'rmsep_percent', 'prediction_time_taken_sec','training_time_taken_sec'])
    if not file_exists:
        writer.writeheader()
    writer.writerows(results)

XGBRegressor: 12.88%

Training time taken for XGBRegressor: 6.7462

prediction time taken for XGBRegressor: 0.0290

RandomForestRegressor: 14.96%

Training time taken for RandomForestRegressor: 65.0368

prediction time taken for RandomForestRegressor: 0.1340

LinearRegression: 23.37%

Training time taken for LinearRegression: 0.6542

prediction time taken for LinearRegression: 0.0062

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008100 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1087
[LightGBM] [Info] Number of data points in the train set: 949194, number of used features: 14
[LightGBM] [Info] Start training from score 5745.395182
LGBMRegressor: 15.38%

Training time taken for LGBMRegressor: 3.2708

prediction time taken for LGBMRegressor: 0.0799

